In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
from PIL import Image

In [3]:
from PIL import Image
import numpy as np
import torch

class ImageDataset(Dataset):
    def __init__(self, dataframe, base_dir, transform=None, train=True):
        self.dataframe = dataframe
        self.base_dir = base_dir
        self.transform = transform
        self.train = train

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        file_name = self.dataframe.iloc[idx]['file_name']
        img_path = os.path.join(self.base_dir, file_name)
        image = Image.open(img_path).convert("RGB")
    
        # Convert to grayscale and add as 4th channel
        gray = np.array(image.convert("L")) / 255.0
        gray = np.expand_dims(gray, axis=2)
        rgb = np.array(image) / 255.0
        img_4ch = np.concatenate((rgb, gray), axis=2).astype(np.float32)
    
        # Convert back to PIL so transforms can work properly
        img_pil = Image.fromarray((img_4ch[:, :, :3] * 255).astype(np.uint8))
    
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
    
        if self.train:
            label = int(self.dataframe.iloc[idx]['label'])
            return img_tensor, label
        else:
            return img_tensor, file_name


In [4]:
train_df = pd.read_csv("/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/train.csv")

transform_vis = transforms.Compose([
    transforms.ToTensor()  # keeps original size
])

train_dataset = ImageDataset(train_df,
                             "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset",
                             transform=transform_vis,
                             train=True)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [5]:
len(train_dataset)

79950

In [6]:
from torchvision import transforms
import torch

# --- Custom Gaussian Noise transform ---
class AddGaussianNoise(object):
    def __init__(self, mean=0., std=0.05):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

    def __repr__(self):
        return f'{self.__class__.__name__}(mean={self.mean}, std={self.std})'


# --- Convert 3-channel (RGB) → 4-channel (RGB + grayscale) ---
class RGBToRGBGray(object):
    def __call__(self, tensor):
        # tensor shape: [3, H, W]
        gray = tensor.mean(dim=0, keepdim=True)  # grayscale channel
        return torch.cat((tensor, gray), dim=0)  # -> [4, H, W]


# --- Train Transform ---
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAutocontrast(p=0.5),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(kernel_size=(3, 5), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    RGBToRGBGray(),  # <- add grayscale as 4th channel
    AddGaussianNoise(mean=0., std=0.02),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406, 0.45],  # add mean for 4th channel
        std=[0.229, 0.224, 0.225, 0.225]   # add std for 4th channel
    )
])

# --- Validation Transform ---
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    RGBToRGBGray(),  # ensure same input shape for validation
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406, 0.45],
        std=[0.229, 0.224, 0.225, 0.225]
    )
])


In [7]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['label'], random_state=42)


In [8]:
train_dataset = ImageDataset(train_df, "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset", transform=train_transform, train=True)
val_dataset   = ImageDataset(val_df, "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset", transform=val_transform, train=True)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [9]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from torchvision.models import swin_t, Swin_T_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------------------------------
# 1️⃣ Define helper functions to modify input channels
# ----------------------------------------------------
def modify_first_conv_layer(model, in_channels=4):
    """Modify ConvNeXt first conv layer to accept 4-channel input."""
    old_conv = model.features[0][0]  # ConvNeXt first conv
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )
    with torch.no_grad():
        new_conv.weight[:, :3] = old_conv.weight  # copy RGB weights
        new_conv.weight[:, 3:] = old_conv.weight[:, :1]  # initialize gray as avg of R
    model.features[0][0] = new_conv
    return model


def modify_swin_input(model, in_channels=4):
    """
    Modify Swin first patch embedding layer to accept 4-channel input.
    Compatible with torchvision >=0.15 where patch_embed is stored as
    model.features[0].  For older versions it’s model.patch_embed.
    """
    patch_embed = None

    # 1️⃣ Check for modern structure
    if hasattr(model, "features"):
        for m in model.features.modules():
            if isinstance(m, nn.Conv2d) and m.in_channels == 3:
                patch_embed = m
                break

    # 2️⃣ Fallback: check older structure (model.patch_embed.proj)
    if patch_embed is None and hasattr(model, "patch_embed"):
        if hasattr(model.patch_embed, "proj"):
            patch_embed = model.patch_embed.proj

    if patch_embed is None:
        raise ValueError("❌ Could not locate Swin patch embedding Conv2d layer")

    old_proj = patch_embed
    new_proj = nn.Conv2d(
        in_channels,
        old_proj.out_channels,
        kernel_size=old_proj.kernel_size,
        stride=old_proj.stride,
        padding=old_proj.padding,
        bias=(old_proj.bias is not None),
    )

    with torch.no_grad():
        # copy pretrained weights for first 3 channels
        new_proj.weight[:, :3] = old_proj.weight
        # duplicate first channel for the new (4th) input
        new_proj.weight[:, 3:] = old_proj.weight[:, :1]
        if old_proj.bias is not None:
            new_proj.bias[:] = old_proj.bias

    # Replace in model
    if hasattr(model, "features") and isinstance(model.features[0], nn.Sequential):
        model.features[0][0] = new_proj
    elif hasattr(model, "patch_embed"):
        model.patch_embed.proj = new_proj

    return model


# ----------------------------------------------------
# 2️⃣ Load pretrained models
# ----------------------------------------------------
conv_weights = ConvNeXt_Tiny_Weights.DEFAULT
convnext = convnext_tiny(weights=conv_weights)
convnext.classifier = nn.Identity()

swin_weights = Swin_T_Weights.DEFAULT
swin = swin_t(weights=swin_weights)
swin.head = nn.Identity()

# ----------------------------------------------------
# 3️⃣ Modify both to accept 4-channel input
# ----------------------------------------------------
convnext = modify_first_conv_layer(convnext, in_channels=4)
swin = modify_swin_input(swin, in_channels=4)

# ----------------------------------------------------
# 4️⃣ Define fusion & combined model (your existing part)
# ----------------------------------------------------
class AttentionFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.ReLU(),
            nn.Linear(dim // 2, 2),
            nn.Softmax(dim=1)
        )

    def forward(self, f1, f2):
        weights = self.attn(torch.cat((f1, f2), dim=1))
        w1, w2 = weights[:, 0].unsqueeze(1), weights[:, 1].unsqueeze(1)
        fused = w1 * f1 + w2 * f2
        return fused


class ConvNeXtSwinFusion(nn.Module):
    def __init__(self, convnext, swin, num_classes=2):
        super().__init__()
        self.convnext = convnext
        self.swin = swin
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fusion = AttentionFusion(dim=768 * 2)
        self.classifier = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        f1 = self.convnext(x)
        f1 = self.pool(f1).flatten(1)
        f2 = self.swin(x)
        fused = self.fusion(f1, f2)
        out = self.classifier(fused)
        return out


# ----------------------------------------------------
# 5️⃣ Instantiate final model
# ----------------------------------------------------
model = ConvNeXtSwinFusion(convnext, swin, num_classes=2).to(device)
print("✅ Model ready for 4-channel (RGB+Gray) input")


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 218MB/s]  


Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 199MB/s]  


✅ Model ready for 4-channel (RGB+Gray) input


In [10]:
criterion = nn.CrossEntropyLoss()       
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [11]:
import sys

num_epochs = 2
BAR_WIDTH = 40

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    total_batches = len(train_loader)

    for batch_idx, (images, labels) in enumerate(train_loader, 1):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Progress bar
        progress = batch_idx / total_batches
        filled = int(BAR_WIDTH * progress)
        bar = "█" * filled + "░" * (BAR_WIDTH - filled)
        sys.stdout.write(
            f"\rEpoch [{epoch+1}/{num_epochs}] |{bar}| {batch_idx}/{total_batches} "
            f"Loss: {running_loss / batch_idx:.4f}"
        )
        sys.stdout.flush()

    print()  # move to next line after epoch completes

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_loss = val_loss / len(val_loader)

    print(f"Training Loss (avg): {running_loss / total_batches:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

Epoch [1/2] |████████████████████████████████████████| 1999/1999 Loss: 0.1758
Training Loss (avg): 0.1758
Validation Loss: 0.2407, Validation Accuracy: 0.8969
Epoch [2/2] |████████████████████████████████████████| 1999/1999 Loss: 0.1115
Training Loss (avg): 0.1115
Validation Loss: 0.1456, Validation Accuracy: 0.9421


In [12]:
#----------------------------------
from sklearn.metrics import classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(
    all_labels,
    all_preds,
    target_names=["Human", "AI"]
))


              precision    recall  f1-score   support

       Human       0.99      0.89      0.94      7995
          AI       0.90      1.00      0.95      7995

    accuracy                           0.94     15990
   macro avg       0.95      0.94      0.94     15990
weighted avg       0.95      0.94      0.94     15990



In [13]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Human", "AI"],
    digits=4
))

              precision    recall  f1-score   support

       Human     0.9947    0.8889    0.9388      7995
          AI     0.8996    0.9952    0.9450      7995

    accuracy                         0.9421     15990
   macro avg     0.9471    0.9421    0.9419     15990
weighted avg     0.9471    0.9421    0.9419     15990



In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [14]:
test_dir = "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/test_data_v2"


class TestImageDataset(Dataset):
    def __init__(self, dataframe, base_dir, transform=None):
        self.dataframe = dataframe
        self.base_dir = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Keep original id (for submission)
        original_id = self.dataframe.iloc[idx]['id']

        # Clean filename only for loading
        file_name = original_id
        if not file_name.endswith(".jpg"):
            file_name = file_name + ".jpg"
        file_name = os.path.basename(file_name)

        img_path = os.path.join(self.base_dir, file_name)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, original_id   # return original id, not filename



In [15]:
test_df = pd.read_csv("/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/test.csv")

test_dir = "/kaggle/input/datasets/alessandrasala79/ai-vs-human-generated-dataset/test_data_v2"

test_dataset = TestImageDataset(
    dataframe=test_df,
    base_dir=test_dir,
    transform=val_transform
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [16]:
model.eval()
preds_list = []
ids = []

with torch.no_grad():
    for images, original_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        preds_list.extend(preds.cpu().numpy())
        ids.extend(original_ids)  # use original test.csv ids

# Create submission (exact same ids as test.csv)
submission = pd.DataFrame({
    "id": ids,
    "label": preds_list
})
submission.to_csv("submission_images_16.csv", index=False)
print('Completed the submission file - 14 file..........')

Completed the submission file - 14 file..........
